# Persistence：PostgreSQL 长期记忆与线程状态

本 Notebook 使用两类 PostgreSQL 持久化组件，并调用项目当前配置的真实 DeepSeek 模型：

- <code>PostgresStore</code>：保存 namespace/key/value 形式的跨调用、跨线程业务记忆；
- <code>PostgresSaver</code>：保存由 <code>thread_id</code> 标识的 Graph State 与 checkpoint 历史。

代码不为配置缺失、数据库断连或模型失败提供回退路径。环境变量直接读取，外部调用异常直接抛出，让失败停在真实发生的位置。输出也只保留能观察运行机制的原始对象或关键字段。

## 1. Store 与 Saver 的职责

| 组件 | 定位方式 | 保存内容 | 跨线程共享 |
|---|---|---|---|
| <code>PostgresStore</code> | <code>namespace + key</code> | 用户档案、偏好、事实等业务记忆 | 可以，由 namespace 中的业务身份控制 |
| <code>PostgresSaver</code> | <code>configurable.thread_id</code> | Graph State、执行步骤与 checkpoint 历史 | 不共享，每个 thread 是独立时间线 |

```mermaid
flowchart LR
    U[同一用户 alice] --> A[调用 A / thread-A]
    U --> B[调用 B / thread-B]
    A --> SA[(PostgresSaver<br/>thread-A State)]
    B --> SB[(PostgresSaver<br/>thread-B State)]
    A --> ST[(PostgresStore<br/>namespace + key + value)]
    B --> ST
    ST --> PG[(PostgreSQL)]
    SA --> PG
    SB --> PG
```

同一个 PostgreSQL 服务可以承载两类数据，但两套 API、表结构与作用域不同。<code>thread_id</code> 不能代替用户 namespace，Store 也不能代替 checkpoint 历史。

## 2. 直接配置

请从项目目录启动 Jupyter，并直接调用 <code>load_dotenv(override=True)</code>。PostgreSQL URI 优先读取 <code>LANGGRAPH_POSTGRES_URI</code>，否则必须存在 <code>LANGCHAIN_POSTGRES_URL</code>；DeepSeek 必须由项目环境提供对应密钥。代码不会输出 URI、API key 或私有主机。

In [9]:
import operator
import os
import subprocess
import sys
from dataclasses import dataclass
from pprint import pformat
from typing import Annotated, Any, TypedDict
from uuid import uuid4

import psycopg
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime
from langgraph.store.postgres import PostgresStore

load_dotenv(override=True)
POSTGRES_URI = (
    os.getenv('LANGGRAPH_POSTGRES_URI')
    or os.environ['LANGCHAIN_POSTGRES_URL']
)
chat_model = init_chat_model(
    model='deepseek:deepseek-v4-flash',
    temperature=0,
    timeout=60,
    max_retries=0,
)

## 3. 首次 setup

首次使用目标数据库/schema 时必须显式执行 <code>PostgresStore.setup()</code> 与 <code>PostgresSaver.setup()</code>。这里不捕获连接或迁移异常：任一服务条件不满足，当前单元会直接失败并保留真实 traceback。setup 本身没有值得重复展示的返回对象，因此本单元不打印成功状态。

In [10]:
with PostgresStore.from_conn_string(POSTGRES_URI) as store:
    store.setup()

with PostgresSaver.from_conn_string(POSTGRES_URI) as saver:
    saver.setup()

## 4. PostgresStore：namespace、key、value

本次运行生成随机用户后缀，避免覆盖已有记录。业务层级是：

<code>('users', user_id, 'memories') + key → value</code>

下面只展示四类有助于理解 Store 的原始结果：完整 namespace 查询、结构化 filter、缺失 key，以及不同用户 namespace 的读取。

In [11]:
DEMO_RUN_ID = uuid4().hex
ALICE_USER_ID = f'alice-{DEMO_RUN_ID}'
BOB_USER_ID = f'bob-{DEMO_RUN_ID}'
alice_namespace = ('users', ALICE_USER_ID, 'memories')
bob_namespace = ('users', BOB_USER_ID, 'memories')

memory_records = {
    'profile': {
        'category': 'profile',
        'text': '用户名字叫小花，主要使用中文。',
    },
    'hobby': {
        'category': 'hobby',
        'text': '周末喜欢跑步，也会练习羽毛球。',
    },
    'workflow': {
        'category': 'workflow',
        'text': '学习代码时偏好先查看完整原始对象，再阅读机制解释。',
    },
}

with PostgresStore.from_conn_string(POSTGRES_URI) as store:
    for key, value in memory_records.items():
        store.put(alice_namespace, key, value)

    alice_items = store.search(alice_namespace)
    workflow_items = store.search(
        alice_namespace, filter={'category': 'workflow'}
    )
    missing_item = store.get(alice_namespace, 'missing-key')
    bob_profile = store.get(bob_namespace, 'profile')

print(
    '【完整 Item 列表】观察同一 namespace 下 key、value 与时间字段：\n'
    + pformat(alice_items, sort_dicts=False)
)
print(
    '【结构化 filter 原始结果】观察 Store 如何按 value 字段筛选：\n'
    + pformat(workflow_items, sort_dicts=False)
)
print(
    '【缺失 key 原始结果】观察不存在的 key 返回 None：\n'
    + pformat(missing_item)
)
print(
    '【不同用户 namespace 原始结果】观察 Bob 不能读取 Alice 的 profile：\n'
    + pformat(bob_profile)
)

assert {item.key for item in alice_items} == set(memory_records)
assert [item.key for item in workflow_items] == ['workflow']
assert missing_item is None
assert bob_profile is None

【完整 Item 列表】观察同一 namespace 下 key、value 与时间字段：
[Item(namespace=['users', 'alice-431a06e019214fc7a9ee771b4f59d28e', 'memories'], key='workflow', value={'text': '学习代码时偏好先查看完整原始对象，再阅读机制解释。', 'category': 'workflow'}, created_at='2026-08-20T13:15:50.767231+00:00', updated_at='2026-08-20T13:15:50.767231+00:00', score=None),
 Item(namespace=['users', 'alice-431a06e019214fc7a9ee771b4f59d28e', 'memories'], key='hobby', value={'text': '周末喜欢跑步，也会练习羽毛球。', 'category': 'hobby'}, created_at='2026-08-20T13:15:50.760773+00:00', updated_at='2026-08-20T13:15:50.760773+00:00', score=None),
 Item(namespace=['users', 'alice-431a06e019214fc7a9ee771b4f59d28e', 'memories'], key='profile', value={'text': '用户名字叫小花，主要使用中文。', 'category': 'profile'}, created_at='2026-08-20T13:15:50.752590+00:00', updated_at='2026-08-20T13:15:50.752590+00:00', score=None)]
【结构化 filter 原始结果】观察 Store 如何按 value 字段筛选：
[Item(namespace=['users', 'alice-431a06e019214fc7a9ee771b4f59d28e', 'memories'], key='workflow', value={'text': '学习代码时偏好先

## 5. 在 StateGraph 中注入 Store 与 Saver

图节点通过 <code>Runtime.store</code> 按用户 namespace 检索长期记忆；编译时通过 <code>checkpointer=...</code> 注入 Saver。

- <code>UserContext.user_id</code> 决定 Store namespace；
- <code>RunnableConfig.configurable.thread_id</code> 决定 Saver 时间线；
- 检索节点把 Store 返回的完整字段转换为可进入 Graph State 的普通字典；
- 模型节点只依据检索到的 canonical 记忆回答。

模型调用没有回退分支，真实错误会直接终止图执行。

In [12]:
class MemoryState(TypedDict, total=False):
    question: str
    events: Annotated[list[str], operator.add]
    retrieved_memories: list[dict[str, Any]]
    model_response: AIMessage
    answer: str


@dataclass(frozen=True)
class UserContext:
    user_id: str


def retrieve_workflow_memory(
    state: MemoryState, runtime: Runtime[UserContext]
) -> MemoryState:
    namespace = ('users', runtime.context.user_id, 'memories')
    return {
        'events': ['retrieve_postgres_store'],
        'retrieved_memories': [
            {
                'namespace': tuple(item.namespace),
                'key': item.key,
                'value': item.value,
                'created_at': item.created_at,
                'updated_at': item.updated_at,
            }
            for item in runtime.store.search(
                namespace, filter={'category': 'workflow'}
            )
        ],
    }


def answer_with_model(state: MemoryState) -> MemoryState:
    response = chat_model.invoke(
        [
            SystemMessage(
                content=(
                    '你是严谨的中文助手。只依据给定长期记忆，'
                    '用一到两句话回答；不要补充记忆中没有的信息。'
                )
            ),
            HumanMessage(
                content=(
                    f'用户问题：{state["question"]}\n'
                    'PostgresStore 检索到的长期记忆：\n'
                    + pformat(
                        state['retrieved_memories'], sort_dicts=False
                    )
                )
            ),
        ]
    )
    return {
        'events': ['call_deepseek'],
        'model_response': response,
        'answer': response.content,
    }


def build_memory_graph(checkpointer, store):
    builder = StateGraph(MemoryState, context_schema=UserContext)
    builder.add_node('retrieve_workflow_memory', retrieve_workflow_memory)
    builder.add_node('answer_with_model', answer_with_model)
    builder.add_edge(START, 'retrieve_workflow_memory')
    builder.add_edge('retrieve_workflow_memory', 'answer_with_model')
    builder.add_edge('answer_with_model', END)
    return builder.compile(checkpointer=checkpointer, store=store)

## 6. 第一次运行：真实 LLM 与 checkpoint

使用 <code>thread-A</code> 运行图。输出保留四个互补层次：

1. <code>AIMessage</code>：观察真实模型响应及其元数据；
2. 最新 <code>StateSnapshot</code>：观察当前线程最终 State、配置和父 checkpoint；
3. checkpoint history：观察每个 superstep 的状态演进；
4. <code>CheckpointTuple</code>：观察 Saver 实际读取到的底层 checkpoint 记录。

In [13]:
THREAD_A_ID = f'postgres-ltm-a-{DEMO_RUN_ID}'
thread_a_config: RunnableConfig = {
    'configurable': {'thread_id': THREAD_A_ID}
}
alice_context = UserContext(user_id=ALICE_USER_ID)

with PostgresStore.from_conn_string(POSTGRES_URI) as store:
    with PostgresSaver.from_conn_string(POSTGRES_URI) as saver:
        graph = build_memory_graph(saver, store)
        thread_a_result = graph.invoke(
            {
                'question': '我学习技术时偏好怎样查看结果？',
                'events': [],
            },
            thread_a_config,
            context=alice_context,
            durability='sync',
        )
        thread_a_snapshot = graph.get_state(thread_a_config)
        thread_a_history = list(graph.get_state_history(thread_a_config))
        thread_a_checkpoint = saver.get_tuple(thread_a_config)

print(
    '【原始 AIMessage】观察真实模型内容、用量和响应元数据：\n'
    + pformat(thread_a_result['model_response'], sort_dicts=False)
)
print(
    '【最新 StateSnapshot】观察 thread-A 最终 State 与 checkpoint 配置：\n'
    + pformat(thread_a_snapshot, sort_dicts=False)
)
print(
    '【完整 checkpoint history】观察 superstep 边界上的状态演进：\n'
    + pformat(thread_a_history, sort_dicts=False)
)
print(
    '【原始 CheckpointTuple】观察 Saver 从 PostgreSQL 读取的底层记录：\n'
    + pformat(thread_a_checkpoint, sort_dicts=False)
)

assert isinstance(thread_a_result['model_response'], AIMessage)
assert thread_a_result['model_response'].content
assert thread_a_snapshot.values['answer'] == thread_a_result['answer']
assert thread_a_history
assert thread_a_checkpoint is not None

【原始 AIMessage】观察真实模型内容、用量和响应元数据：
AIMessage(content='您偏好先查看完整原始对象，再阅读机制解释。', additional_kwargs={'refusal': None, 'reasoning_content': '我们根据记忆回答：偏好先查看完整原始对象，再阅读机制解释。'}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 295, 'total_tokens': 326, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 17, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 167}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '22cff51b-0931-4bc4-9a60-7ad4c351d554', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a01f50-4b4e-7ec3-8631-55ef3d46f4c8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 295, 'output_tokens': 31, 'total_tokens': 326, 'input_token_details': {'cache_read': 128}, 'output_token_det

## 7. 新对象与新线程：两种作用域

上一节的 Store/Saver 连接已经关闭。下面创建全新的 Store、Saver 和编译图：

- 新 Store 仍能按 Alice namespace 读取完整 <code>Item</code>；
- 新 Saver 仍能按 <code>thread-A</code> 恢复已有 <code>StateSnapshot</code>；
- 新 <code>thread-B</code> 拥有独立 State，但因为 user_id 仍是 Alice，可以读取同一 Store 记忆并再次调用真实模型。

In [14]:
THREAD_B_ID = f'postgres-ltm-b-{DEMO_RUN_ID}'
thread_b_config: RunnableConfig = {
    'configurable': {'thread_id': THREAD_B_ID}
}

with PostgresStore.from_conn_string(POSTGRES_URI) as store:
    with PostgresSaver.from_conn_string(POSTGRES_URI) as saver:
        reopened_graph = build_memory_graph(saver, store)
        reopened_profile = store.get(alice_namespace, 'profile')
        restored_thread_a = reopened_graph.get_state(thread_a_config)
        thread_b_result = reopened_graph.invoke(
            {
                'question': '我学习技术时偏好怎样查看结果？',
                'events': [],
            },
            thread_b_config,
            context=alice_context,
            durability='sync',
        )
        thread_b_snapshot = reopened_graph.get_state(thread_b_config)

print(
    '【新 Store 对象读取的 Item】观察业务记忆不依赖旧 Python 对象：\n'
    + pformat(reopened_profile, sort_dicts=False)
)
print(
    '【新 Saver 对象恢复的 thread-A】观察 checkpoint 按原 thread_id 恢复：\n'
    + pformat(restored_thread_a, sort_dicts=False)
)
print(
    '【thread-B 原始 AIMessage】观察新线程复用 Alice 长期记忆后的真实回答：\n'
    + pformat(thread_b_result['model_response'], sort_dicts=False)
)
print(
    '【thread-B StateSnapshot】观察新线程拥有独立 checkpoint 时间线：\n'
    + pformat(thread_b_snapshot, sort_dicts=False)
)

assert reopened_profile.value == memory_records['profile']
assert restored_thread_a.values == thread_a_snapshot.values
assert isinstance(thread_b_result['model_response'], AIMessage)
assert thread_b_snapshot.config['configurable']['thread_id'] == THREAD_B_ID
assert restored_thread_a.config['configurable']['thread_id'] == THREAD_A_ID

【新 Store 对象读取的 Item】观察业务记忆不依赖旧 Python 对象：
Item(namespace=['users', 'alice-431a06e019214fc7a9ee771b4f59d28e', 'memories'], key='profile', value={'text': '用户名字叫小花，主要使用中文。', 'category': 'profile'}, created_at='2026-08-20T13:15:50.752590+00:00', updated_at='2026-08-20T13:15:50.752590+00:00')
【新 Saver 对象恢复的 thread-A】观察 checkpoint 按原 thread_id 恢复：
StateSnapshot(values={'question': '我学习技术时偏好怎样查看结果？', 'events': ['retrieve_postgres_store', 'call_deepseek'], 'retrieved_memories': [{'namespace': ['users', 'alice-431a06e019214fc7a9ee771b4f59d28e', 'memories'], 'key': 'workflow', 'value': {'text': '学习代码时偏好先查看完整原始对象，再阅读机制解释。', 'category': 'workflow'}, 'created_at': datetime.datetime(2026, 8, 20, 13, 15, 50, 767231, tzinfo=datetime.timezone.utc), 'updated_at': datetime.datetime(2026, 8, 20, 13, 15, 50, 767231, tzinfo=datetime.timezone.utc)}], 'model_response': AIMessage(content='您偏好先查看完整原始对象，再阅读机制解释。', additional_kwargs={'refusal': None, 'reasoning_content': '我们根据记忆回答：偏好先查看完整原始对象，再阅读机制解释。'}, response

## 8. 新 Python 进程读取

跨对象仍发生在同一解释器。下面启动全新的 Python 进程，只凭相同数据库、namespace/key 与 <code>thread_id</code>，读取完整 <code>Item</code> 和 <code>CheckpointTuple</code>。子进程或外部服务失败会由 <code>check=True</code> 直接抛出。

这验证的是应用进程重启语义；Notebook 不会重启 PostgreSQL 服务，因此不声称验证数据库服务自身的重启恢复。

In [15]:
child_code = r'''
import os
from pprint import pformat

from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore

load_dotenv(override=True)
uri = os.getenv('LANGGRAPH_POSTGRES_URI') or os.environ['LANGCHAIN_POSTGRES_URL']
namespace = ('users', os.environ['LTM_USER_ID'], 'memories')
config = {'configurable': {'thread_id': os.environ['LTM_THREAD_ID']}}

with PostgresStore.from_conn_string(uri) as store:
    item = store.get(namespace, 'profile')
with PostgresSaver.from_conn_string(uri) as saver:
    checkpoint = saver.get_tuple(config)

assert item is not None
assert checkpoint is not None
print(
    '【子进程读取的完整 Item】验证应用进程更换后业务记忆仍在：\n'
    + pformat(item, sort_dicts=False)
)
print(
    '【子进程读取的 CheckpointTuple】验证应用进程更换后线程记录仍在：\n'
    + pformat(checkpoint, sort_dicts=False)
)
'''

child_environment = os.environ.copy()
child_environment['LTM_USER_ID'] = ALICE_USER_ID
child_environment['LTM_THREAD_ID'] = THREAD_A_ID
child_result = subprocess.run(
    [sys.executable, '-c', child_code],
    cwd=os.getcwd(),
    env=child_environment,
    capture_output=True,
    text=True,
    timeout=30,
    check=True,
)
print(
    '【新 Python 进程的原始标准输出】观察两个 PostgreSQL 持久对象：\n'
    + child_result.stdout
)

【新 Python 进程的原始标准输出】观察两个 PostgreSQL 持久对象：
【子进程读取的完整 Item】验证应用进程更换后业务记忆仍在：
Item(namespace=['users', 'alice-431a06e019214fc7a9ee771b4f59d28e', 'memories'], key='profile', value={'text': '用户名字叫小花，主要使用中文。', 'category': 'profile'}, created_at='2026-08-20T13:15:50.752590+00:00', updated_at='2026-08-20T13:15:50.752590+00:00')
【子进程读取的 CheckpointTuple】验证应用进程更换后线程记录仍在：
CheckpointTuple(config={'configurable': {'thread_id': 'postgres-ltm-a-431a06e019214fc7a9ee771b4f59d28e', 'checkpoint_ns': '', 'checkpoint_id': '1f19c994-46a5-65c8-8002-0af9851ae097'}}, checkpoint={'v': 4, 'id': '1f19c994-46a5-65c8-8002-0af9851ae097', 'ts': '2026-08-20T13:15:51.694479+00:00', 'versions_seen': {'__input__': {}, '__start__': {'__start__': '00000000000000000000000000000001.0.6805824403238611'}, 'answer_with_model': {'branch:to:answer_with_model': '00000000000000000000000000000003.0.9139667462141552'}, 'retrieve_workflow_memory': {'branch:to:retrieve_workflow_memory': '00000000000000000000000000000002.0.195902427577061

## 9. PostgreSQL 表中的原始证据

公开 API 已经展示业务行为。最后用参数化只读 SQL 展示本次 namespace 在 <code>store</code> 表中的原始行，以及两个 thread 在 <code>checkpoints</code> 表中的实际记录。这里不打印行数或成功标签。

In [16]:
with psycopg.connect(POSTGRES_URI, connect_timeout=5) as connection:
    store_rows = connection.execute(
        '''
        SELECT prefix, key, value, created_at, updated_at
        FROM store
        WHERE prefix = %s
        ORDER BY key
        ''',
        ('.'.join(alice_namespace),),
    ).fetchall()
    checkpoint_rows = connection.execute(
        '''
        SELECT thread_id, checkpoint_id, parent_checkpoint_id, metadata
        FROM checkpoints
        WHERE thread_id IN (%s, %s)
        ORDER BY thread_id, checkpoint_id
        ''',
        (THREAD_A_ID, THREAD_B_ID),
    ).fetchall()

print(
    '【store 表原始行】观察 namespace/key/value 的 PostgreSQL 落盘形态：\n'
    + pformat(store_rows, sort_dicts=False)
)
print(
    '【checkpoints 表原始行】观察 thread_id、父子 checkpoint 与 metadata：\n'
    + pformat(checkpoint_rows, sort_dicts=False)
)

【store 表原始行】观察 namespace/key/value 的 PostgreSQL 落盘形态：
[('users.alice-431a06e019214fc7a9ee771b4f59d28e.memories',
  'hobby',
  {'text': '周末喜欢跑步，也会练习羽毛球。', 'category': 'hobby'},
  datetime.datetime(2026, 8, 20, 13, 15, 50, 760773, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC')),
  datetime.datetime(2026, 8, 20, 13, 15, 50, 760773, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC'))),
 ('users.alice-431a06e019214fc7a9ee771b4f59d28e.memories',
  'profile',
  {'text': '用户名字叫小花，主要使用中文。', 'category': 'profile'},
  datetime.datetime(2026, 8, 20, 13, 15, 50, 752590, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC')),
  datetime.datetime(2026, 8, 20, 13, 15, 50, 752590, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC'))),
 ('users.alice-431a06e019214fc7a9ee771b4f59d28e.memories',
  'workflow',
  {'text': '学习代码时偏好先查看完整原始对象，再阅读机制解释。', 'category': 'workflow'},
  datetime.datetime(2026, 8, 20, 13, 15, 50, 767231, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC')),
  datetime.datetime(2026, 8, 20, 13, 15, 50, 767231, tzinfo=zoneinfo.ZoneInfo(key='

## 10. 结果解释与一致性边界

- 新 Store 对象和新 Python 进程仍能读取 <code>Item</code>，说明业务记忆来自 PostgreSQL，而不是旧对象；
- 新 Saver 对象和新 Python 进程仍能按 <code>thread-A</code> 读取 checkpoint，说明线程状态同样来自 PostgreSQL；
- <code>thread-B</code> 没有继承 <code>thread-A</code> 的 State，但两者使用同一 Alice namespace，因此都能读取同一长期记忆；
- Bob namespace 和缺失 key 返回 <code>None</code>，说明用户 namespace 与 thread 时间线是两套独立边界；
- Store 写入与 Saver checkpoint 不是一个跨表事务。生产系统若要求两者强一致，需要设计业务事务、幂等键、补偿或 outbox；
- 本 Notebook 验证了跨调用、跨 Store/Saver 对象和跨应用 Python 进程读取，但没有重启 PostgreSQL 服务，也没有验证备份恢复或故障切换。

## 11. 生产边界

1. namespace 中的用户身份必须来自可信认证上下文，不能直接相信用户文本；
2. Store value 与 checkpoint 都可能包含敏感信息，应落实 TLS、最小权限、审计、备份与删除策略；
3. <code>setup()</code> 更适合部署迁移阶段，不应让每个线上请求重复执行；
4. 是否写入长期记忆不应只依赖模型提示词，关键写入应由确定性应用逻辑或中间件控制；
5. 外部模型调用和节点副作用需要合理超时、幂等键与可观测性；
6. 本教程使用同步 API；异步应用应选择对应 async API 与受控连接池。

## 12. 总结

<code>PostgresStore</code> 负责可跨线程共享的业务记忆，<code>PostgresSaver</code> 负责按 <code>thread_id</code> 隔离的 Graph State 历史。判断数据是否来自持久后端，不看成功口号或布尔标签，而看新对象、新线程、新 Python 进程实际读回的 <code>Item</code>、<code>StateSnapshot</code> 与 <code>CheckpointTuple</code>。